In [1]:
from pathlib import Path

import yaml
import pandas as pd
import json

In [2]:
pd.set_option("display.max_columns", None)      # 모든 column 표시
pd.set_option("display.max_rows", None)         # 모든 row 표시
pd.set_option("display.max_colwidth", None)     # column 안의 문자열 생략 안 함
pd.set_option("display.width", None)            # 출력 폭 제한 완화
pd.set_option("display.expand_frame_repr", False)  # 여러 줄로 쪼개서 출력하지 않음

In [3]:
SUITE_ROOT = Path("generated_suites")
FPGA_BIN_LABELS = (
  "improve_no_tcu_lut_fexp",
  "improve_tcol32",
  "naive_gemm_tcol32",
  "naive_simd",
)


def load_yaml(path):
  path = Path(path)
  with path.open("r", encoding="utf-8") as f:
    return yaml.safe_load(f)


def load_merged_suites(suite_group, stage):
  merged_dir = SUITE_ROOT / suite_group / f"{stage}_merged"
  return {
    label: load_yaml(merged_dir / f"{stage}_merged_{label}.yaml")
    for label in FPGA_BIN_LABELS
  }


main_prefill = load_merged_suites("main", "prefill")
main_generation = load_merged_suites("main", "generation")
main_all_prefill = load_merged_suites("main_all", "prefill")
main_all_generation = load_merged_suites("main_all", "generation")

In [4]:
improve_tcol32_df = pd.read_csv("outputs_main/improve_tcol32/raw_db.csv")
improve_no_tcu_lut_fexp_df = pd.read_csv("outputs_main/improve_no_tcu_lut_fexp/raw_db.csv")
naive_gemm_tcol32_df = pd.read_csv("outputs_main/naive_gemm_tcol32/raw_db.csv")
naive_simd_df = pd.read_csv("outputs_main/naive_simd/raw_db.csv")

improve_no_tcu_lut_fexp = improve_no_tcu_lut_fexp_df
naive_gemm_tcol32 = naive_gemm_tcol32_df
naive_simd = naive_simd_df

In [5]:
def suites_for_bin(fpga_bin_label, target_main_all=False):
  """Return prefill+generation merged suites for one FPGA bin."""
  prefill = main_all_prefill if target_main_all else main_prefill
  generation = main_all_generation if target_main_all else main_generation
  return [prefill[fpga_bin_label], generation[fpga_bin_label]]


def _suite_items(suites):
  if isinstance(suites, dict) and "cases" in suites:
    return [(suites.get("name", "suite"), suites)]
  if isinstance(suites, dict):
    return list(suites.items())
  return [(suite.get("name", f"suite_{idx}"), suite) for idx, suite in enumerate(suites)]


def _normalize_shape(value):
  if value is None:
    return None
  if isinstance(value, str):
    try:
      value = json.loads(value)
    except json.JSONDecodeError:
      return value
  if isinstance(value, dict):
    return json.dumps(value, sort_keys=True)
  try:
    if pd.isna(value):
      return None
  except (TypeError, ValueError):
    pass
  return str(value)


def _expected_cases(suites):
  rows = []
  for suite_label, suite in _suite_items(suites):
    defaults = suite.get("defaults", {})
    suite_name = suite.get("name", suite_label)
    for case in suite.get("cases", []):
      rows.append({
        "case_id": case.get("id"),
        "expected_suite": suite_name,
        "expected_fpga_bin": defaults.get("fpga_bin"),
        "expected_app": case.get("app", defaults.get("app")),
        "expected_kind": case.get("kind"),
        "expected_stage": case.get("stage"),
        "expected_name": case.get("name"),
        "expected_args": case.get("args"),
        "expected_shape_key": _normalize_shape(case.get("shape")),
      })
  return pd.DataFrame(rows)


def summarize(df, suites=None, target_main_all=False):
  """
  Summarize raw_db status and compare observed case IDs with generated suites.

  If suites is omitted, the function infers the FPGA bin from df and selects
  main or main_all according to target_main_all. Use target_main_all=True for
  fpint GEMM bins and False for the non-GEMM main runs.
  """
  observed = df.copy()
  if suites is None:
    labels = observed["fpga_bin_label"].dropna().astype(str).unique()
    if len(labels) != 1:
      raise ValueError(f"expected one fpga_bin_label, got {labels}")
    suites = suites_for_bin(labels[0], target_main_all=target_main_all)

  expected = _expected_cases(suites)
  if expected.empty:
    raise ValueError("no expected cases found in suites")
  if "case_id" not in observed.columns:
    raise ValueError("raw DB is missing required case_id column")

  if "shape_json" in observed.columns:
    observed["shape_key"] = observed["shape_json"].map(_normalize_shape)
  sort_cols = [c for c in ["case_id", "timestamp_utc", "run_id"] if c in observed.columns]
  latest = observed.sort_values(sort_cols).drop_duplicates("case_id", keep="last") if sort_cols else observed.drop_duplicates("case_id", keep="last")

  expected_ids = set(expected["case_id"].dropna())
  observed_ids = set(observed["case_id"].dropna())
  missing_ids = expected_ids - observed_ids
  unexpected_ids = observed_ids - expected_ids
  duplicate_case_counts = observed["case_id"].value_counts(dropna=False)
  duplicate_ids = set(duplicate_case_counts[duplicate_case_counts > 1].index.dropna())

  status_counts = (
    observed["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="rows")
    if "status" in observed.columns else pd.DataFrame(columns=["status", "rows"])
  )
  pass_rows = int(status_counts.loc[status_counts["status"] == "pass", "rows"].sum()) if not status_counts.empty else 0
  observed_cases = int(observed["case_id"].nunique(dropna=True))

  overview = pd.DataFrame([{
    "expected_cases": len(expected),
    "observed_rows": len(observed),
    "observed_cases": observed_cases,
    "pass_rows": pass_rows,
    "nonpass_rows": len(observed) - pass_rows,
    "missing_cases": len(missing_ids),
    "unexpected_cases": len(unexpected_ids),
    "duplicate_case_ids": len(duplicate_ids),
    "row_count_ok": len(observed) == len(expected),
    "case_set_ok": not missing_ids and not unexpected_ids,
  }])

  expected_by_suite = expected.groupby("expected_suite").size().rename("expected_cases")
  observed_by_suite = observed.groupby("suite")["case_id"].agg(observed_rows="size", observed_cases="nunique") if "suite" in observed.columns else pd.DataFrame()
  by_suite = (
    expected_by_suite.to_frame()
    .join(observed_by_suite, how="outer")
    .fillna(0)
    .astype(int)
    .reset_index()
    .rename(columns={"expected_suite": "suite", "index": "suite"})
  )

  observed_cols = [
    c for c in ["case_id", "suite", "status", "failure_phase", "failure_reason", "app", "kind", "stage", "name", "args", "shape_key", "log_file", "p50_us", "fpga_cycle_min"]
    if c in latest.columns
  ]
  comparison = expected.merge(latest[observed_cols], on="case_id", how="outer", indicator=True)
  missing = comparison[comparison["case_id"].isin(missing_ids)].sort_values(["expected_suite", "case_id"])
  unexpected = comparison[comparison["case_id"].isin(unexpected_ids)].sort_values(["suite", "case_id"] if "suite" in comparison.columns else ["case_id"])
  failures = observed[observed["status"].astype(str) != "pass"].copy() if "status" in observed.columns else pd.DataFrame()
  duplicates = observed[observed["case_id"].isin(duplicate_ids)].sort_values(sort_cols) if duplicate_ids and sort_cols else pd.DataFrame()

  mismatch_masks = []
  for expected_col, observed_col in [
    ("expected_suite", "suite"),
    ("expected_app", "app"),
    ("expected_kind", "kind"),
    ("expected_stage", "stage"),
    ("expected_name", "name"),
    ("expected_args", "args"),
    ("expected_shape_key", "shape_key"),
  ]:
    if expected_col in comparison.columns and observed_col in comparison.columns:
      mismatch_masks.append(
        (comparison["_merge"] == "both")
        & comparison[expected_col].notna()
        & comparison[observed_col].notna()
        & (comparison[expected_col].astype(str) != comparison[observed_col].astype(str))
      )
  metadata_mismatches = comparison[pd.concat(mismatch_masks, axis=1).any(axis=1)] if mismatch_masks else pd.DataFrame()

  return {
    "overview": overview,
    "status_counts": status_counts,
    "by_suite": by_suite,
    "missing": missing,
    "unexpected": unexpected,
    "failures": failures,
    "duplicates": duplicates,
    "metadata_mismatches": metadata_mismatches,
    "comparison": comparison,
  }

In [6]:
summary_results = {
  "improve_no_tcu_lut_fexp": summarize(improve_no_tcu_lut_fexp_df),
  "improve_tcol32": summarize(improve_tcol32_df, target_main_all=True),
  "naive_gemm_tcol32": summarize(naive_gemm_tcol32_df, target_main_all=True),
  "naive_simd": summarize(naive_simd_df),
}

summary_overview = pd.concat(
  [result["overview"].assign(fpga_bin=label) for label, result in summary_results.items()],
  ignore_index=True,
)
summary_overview[
  [
    "fpga_bin",
    "expected_cases",
    "observed_rows",
    "observed_cases",
    "pass_rows",
    "nonpass_rows",
    "missing_cases",
    "unexpected_cases",
    "duplicate_case_ids",
    "row_count_ok",
    "case_set_ok",
  ]
]

,fpga_bin,expected_cases,observed_rows,observed_cases,pass_rows,nonpass_rows,missing_cases,unexpected_cases,duplicate_case_ids,row_count_ok,case_set_ok
0,improve_no_tcu_lut_fexp,16,16,16,16,0,0,0,0,True,True
1,improve_tcol32,84,84,84,80,4,0,0,0,True,True
2,naive_gemm_tcol32,84,84,84,71,13,0,0,0,True,True
3,naive_simd,205,205,205,193,12,0,0,0,True,True


In [7]:
nonpass_columns = [
  "fpga_bin_label",
  "suite",
  "case_id",
  "status",
  "returncode",
  "failure_phase",
  "failure_reason",
  "app",
  "kind",
  "stage",
  "name",
  "args",
  "shape_json",
  "elapsed_wall_s",
  "samples",
  "p50_us",
  "fpga_cycle_min",
  "fpga_cycle_parse_error",
  "log_file",
]

failure_frames = [
  result["failures"].assign(summary_fpga_bin=label)
  for label, result in summary_results.items()
  if not result["failures"].empty
]
if failure_frames:
  nonpass_rows = pd.concat(failure_frames, ignore_index=True)
else:
  nonpass_rows = pd.DataFrame(columns=["summary_fpga_bin", *nonpass_columns])

nonpass_dump_columns = ["summary_fpga_bin"] + [
  col for col in nonpass_columns if col in nonpass_rows.columns
]
nonpass_rows = nonpass_rows[nonpass_dump_columns]
nonpass_sort_columns = [
  col for col in ["summary_fpga_bin", "suite", "stage", "name", "case_id"]
  if col in nonpass_rows.columns
]
if nonpass_sort_columns:
  nonpass_rows = nonpass_rows.sort_values(nonpass_sort_columns, kind="stable")
nonpass_rows = nonpass_rows.reset_index(drop=True)

print(f"nonpass rows: {len(nonpass_rows)}")
nonpass_rows

nonpass rows: 29


,summary_fpga_bin,fpga_bin_label,suite,case_id,status,returncode,failure_phase,failure_reason,app,kind,stage,name,args,shape_json,elapsed_wall_s,samples,p50_us,fpga_cycle_min,fpga_cycle_parse_error,log_file
0,improve_tcol32,improve_tcol32,prefill_merged_improve_tcol32,llama3_8b_prefill_C4_alone_batch4_prefill_seq_len32768_qblk32_prefill_down_proj,timeout,124,run,timeout,fpint_gemm_ffn_hw,gemm,prefill,down_proj,-m 131072 -n 4096 -k 14336 -q 32 -t 0 -d 0,"{""K"": 14336, ""M"": 131072, ""N"": 4096, ""QBLK"": 32, ""QDIR"": 0, ""WTRANS"": 0}",604.613,NaN,NaN,NaN,missing_fpga_cycle,/home/jaeyongjang/project.local/vortex/analysis_workspace/latency_on_hw/outputs_main/improve_tcol32/runs/20260629T132231632801Z/logs/33f723eda6.log
1,improve_tcol32,improve_tcol32,prefill_merged_improve_tcol32,llama3_8b_prefill_C4_alone_batch8_prefill_seq_len32768_qblk32_prefill_down_proj,timeout,124,run,timeout,fpint_gemm_ffn_hw,gemm,prefill,down_proj,-m 262144 -n 4096 -k 14336 -q 32 -t 0 -d 0,"{""K"": 14336, ""M"": 262144, ""N"": 4096, ""QBLK"": 32, ""QDIR"": 0, ""WTRANS"": 0}",605.156,NaN,NaN,NaN,missing_fpga_cycle,/home/jaeyongjang/project.local/vortex/analysis_workspace/latency_on_hw/outputs_main/improve_tcol32/runs/20260629T132231632801Z/logs/fe7aa0e4ac.log
2,improve_tcol32,improve_tcol32,prefill_merged_improve_tcol32,llama3_8b_prefill_C4_alone_batch4_prefill_seq_len32768_qblk32_prefill_gate_proj,timeout,124,run,timeout,fpint_gemm_ffn_hw,gemm,prefill,gate_proj,-m 131072 -n 14336 -k 4096 -q 32 -t 0 -d 0,"{""K"": 4096, ""M"": 131072, ""N"": 14336, ""QBLK"": 32, ""QDIR"": 0, ""WTRANS"": 0}",603.742,NaN,NaN,NaN,missing_fpga_cycle,/home/jaeyongjang/project.local/vortex/analysis_workspace/latency_on_hw/outputs_main/improve_tcol32/runs/20260629T132231632801Z/logs/5358c8d580.log
3,improve_tcol32,improve_tcol32,prefill_merged_improve_tcol32,llama3_8b_prefill_C4_alone_batch8_prefill_seq_len32768_qblk32_prefill_gate_proj,timeout,124,run,timeout,fpint_gemm_ffn_hw,gemm,prefill,gate_proj,-m 262144 -n 14336 -k 4096 -q 32 -t 0 -d 0,"{""K"": 4096, ""M"": 262144, ""N"": 14336, ""QBLK"": 32, ""QDIR"": 0, ""WTRANS"": 0}",605.788,NaN,NaN,NaN,missing_fpga_cycle,/home/jaeyongjang/project.local/vortex/analysis_workspace/latency_on_hw/outputs_main/improve_tcol32/runs/20260629T132231632801Z/logs/18cc30b9ac.log
4,naive_gemm_tcol32,naive_gemm_tcol32,prefill_merged_naive_gemm_tcol32,llama3_8b_prefill_C3_batch1_prefill_seq_len32768_qblk32_prefill_attn_qkT,timeout,124,run,timeout,fpint_gemm_ffn_hw_naive,gemm,prefill,attn_qkT,-m 32768 -n 32768 -k 128 -q 128 -t 1 -d 0,"{""K"": 128, ""M"": 32768, ""N"": 32768, ""QBLK"": 128, ""QDIR"": 0, ""WTRANS"": 1, ""per_head"": true}",602.351,NaN,NaN,NaN,missing_fpga_cycle,/home/jaeyongjang/project.local/vortex/analysis_workspace/latency_on_hw/outputs_main/naive_gemm_tcol32/runs/20260629T140256813896Z/logs/9589a31c91.log
5,naive_gemm_tcol32,naive_gemm_tcol32,prefill_merged_naive_gemm_tcol32,llama3_8b_prefill_C2_batch1_prefill_seq_len32768_qblk32_prefill_down_proj,timeout,124,run,timeout,fpint_gemm_ffn_hw_naive,gemm,prefill,down_proj,-m 32768 -n 4096 -k 14336 -q 32 -t 0 -d 0,"{""K"": 14336, ""M"": 32768, ""N"": 4096, ""QBLK"": 32, ""QDIR"": 0, ""WTRANS"": 0}",603.545,NaN,NaN,NaN,missing_fpga_cycle,/home/jaeyongjang/project.local/vortex/analysis_workspace/latency_on_hw/outputs_main/naive_gemm_tcol32/runs/20260629T140256813896Z/logs/dd6d586eac.log
6,naive_gemm_tcol32,naive_gemm_tcol32,prefill_merged_naive_gemm_tcol32,llama3_8b_prefill_C2_batch2_prefill_seq_len32768_qblk32_prefill_down_proj,timeout,124,run,timeout,fpint_gemm_ffn_hw_naive,gemm,prefill,down_proj,-m 65536 -n 4096 -k 14336 -q 32 -t 0 -d 0,"{""K"": 14336, ""M"": 65536, ""N"": 4096, ""QBLK"": 32, ""QDIR"": 0, ""WTRANS"": 0}",603.946,NaN,NaN,NaN,missing_fpga_cycle,/home/jaeyongjang/project.local/vortex/analysis_workspace/latency_on_hw/outputs_main/naive_gemm_tcol32/runs/20260629T140256813896Z/logs/b7473d68d7.log
7,naive_gemm_tcol32,naive_gemm_tcol32,prefill_merged_naive_gemm_tcol32,llama3